<a href="https://colab.research.google.com/github/MihaiDogariu/CV3/blob/main/laborator/CV_3_Lab_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecție single-stage și segmentarea imaginilor
Partea I compară un detector two-stage (Faster R-CNN, laboratorul 6) cu detectoare single-stage (YOLO26) în două aplicații: una în care contează acuratețea (mAP) și una în care contează timpul de inferență.

Partea II tratează segmentarea:
1. antrenăm de la zero o rețea U-Net pe Oxford-IIIT Pet și măsurăm cât costă adnotarea manuală;
2. construim o aplicație de fundal virtual (ca în MS Teams / Zoom), care rulează live pe camera laptopului;
3. separăm persoanele (segmentare de instanțe și panoptică) și le urmărim în timp (tracking, laboratorul 6).

## 0. Configurare
- `EVAL_IMAGES`: numărul de imagini COCO val2017 pe care calculăm mAP (setul complet are 5.000);
- `CPU_IMAGES`: numărul de imagini pe care măsurăm latența pe CPU (Faster R-CNN are nevoie de câteva secunde pe imagine);
- `RETRAIN`, `EPOCHS`, `IMG_SIZE`, `BATCH_SIZE`: antrenarea U-Net din secțiunea 6; cu `RETRAIN = False` se încarcă modelul salvat în `./checkpoints`.

Pachete suplimentare: `ultralytics` (YOLO, tracking), `pycocotools` (mAP), `imageio-ffmpeg` (scrierea video), `gradio` (interfețe web în notebook).

Laboratorul are nevoie de GPU (în Colab: Runtime -> Change runtime type -> GPU).

In [ ]:
import collections
import contextlib
import importlib.util
import io
import os
import random
import sys
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

for package, module in [('ultralytics', 'ultralytics'), ('pycocotools', 'pycocotools'),
                        ('imageio-ffmpeg', 'imageio_ffmpeg'), ('gradio', 'gradio')]:
    if importlib.util.find_spec(module) is None:
        %pip install -q {package}

import cv2
import gradio as gr
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Video, display
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.models.detection import (FasterRCNN_ResNet50_FPN_V2_Weights, MaskRCNN_ResNet50_FPN_V2_Weights,
                                          fasterrcnn_resnet50_fpn_v2, maskrcnn_resnet50_fpn_v2)
from torchvision.models.segmentation import DeepLabV3_MobileNet_V3_Large_Weights, deeplabv3_mobilenet_v3_large
from torchvision.ops import box_iou
from tqdm.auto import tqdm
from ultralytics import YOLO


# Alegem configurația sistemului (cpu/gpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
IN_COLAB = 'google.colab' in sys.modules


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)

DATA_ROOT = './data'
DATA_DIR = './data/lab7'
OUT_DIR = os.path.join(DATA_DIR, 'output')
CKPT_DIR = './checkpoints'
for d in [DATA_DIR, OUT_DIR, CKPT_DIR, os.path.join(DATA_DIR, 'val2017')]:
    os.makedirs(d, exist_ok=True)

EVAL_IMAGES = 500
CPU_IMAGES = 5

RETRAIN = False
EPOCHS = 30
IMG_SIZE = 128
BATCH_SIZE = 32

### 0.1 Descărcarea resurselor
Adnotările COCO val2017 (doar `instances_val2017.json` din arhivă), primele `EVAL_IMAGES` imagini și două videoclipuri: o stradă (Partea I) și un hol în care persoane se apropie de cameră (Partea II).

In [ ]:
ANN_PATH = os.path.join(DATA_DIR, 'instances_val2017.json')
if not os.path.exists(ANN_PATH):
    zip_path = os.path.join(DATA_DIR, 'annotations_trainval2017.zip')
    torch.hub.download_url_to_file('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', zip_path)
    with zipfile.ZipFile(zip_path) as archive, open(ANN_PATH, 'wb') as f:
        f.write(archive.read('annotations/instances_val2017.json'))
    os.remove(zip_path)

with contextlib.redirect_stdout(io.StringIO()):  # pycocotools afișează mesaje la încărcare
    coco = COCO(ANN_PATH)
IMAGE_IDS = sorted(coco.getImgIds())[:EVAL_IMAGES]


def coco_path(image_id):
    # descarcă imaginea (o singură dată) și întoarce calea locală
    name = coco.imgs[image_id]['file_name']
    path = os.path.join(DATA_DIR, 'val2017', name)
    if not os.path.exists(path):
        torch.hub.download_url_to_file('http://images.cocodataset.org/val2017/' + name, path, progress=False)
    return path


def load_image(path):
    # Lucrăm cu imagini RGB, ca tablouri numpy uint8 de dimensiune H x W x 3
    return np.array(Image.open(path).convert('RGB'))


with ThreadPoolExecutor(16) as pool:
    list(tqdm(pool.map(coco_path, IMAGE_IDS), total=len(IMAGE_IDS), desc='COCO val2017'))

VIDEO_URL = 'https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/{}'
VIDEO_STREET = os.path.join(DATA_DIR, 'person-bicycle-car-detection.mp4')
VIDEO_CALL = os.path.join(DATA_DIR, 'face-demographics-walking-and-pause.mp4')
for path in [VIDEO_STREET, VIDEO_CALL]:
    if not os.path.exists(path):
        torch.hub.download_url_to_file(VIDEO_URL.format(os.path.basename(path)), path)
    print('{:<45} {:>5.1f} MB'.format(os.path.basename(path), os.path.getsize(path) / 2**20))

print('{} imagini COCO, {} adnotări'.format(len(IMAGE_IDS), len(coco.getAnnIds(imgIds=IMAGE_IDS))))

### 0.2 Funcții auxiliare
Ca în laboratorul 6. `write_video` scrie fișiere WebM (VP9), pe care le pot reda browserul, Colab și VS Code.

In [ ]:
COCO_CLASSES = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT.meta['categories']  # 91 de poziții: indicele = category_id din COCO
PALETTE = (np.array(plt.cm.tab20.colors) * 255).astype(int).tolist()


def class_color(label):
    return tuple(PALETTE[COCO_CLASSES.index(label) % len(PALETTE)])


def draw_detections(image, det, colors=None):
    # colors: o culoare (r, g, b) pentru fiecare detecție; None = culoarea clasei; o detecție cu culoarea None nu se desenează
    img = image.copy()
    size = max(img.shape[:2])
    thickness = max(1, round(size / 500))
    font_scale = max(0.3, size / 1600)
    if colors is None:
        colors = [class_color(label) for label in det['labels']]
    for box, label, score, color in zip(det['boxes'].round().int().tolist(), det['labels'], det['scores'].tolist(), colors):
        if color is None:
            continue
        x1, y1, x2, y2 = box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(img, '{} {:.2f}'.format(label, score), (x1, max(y1 - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 1, cv2.LINE_AA)
    return img


def show_images(imgs, titles, ncols=3, width=6):
    nrows = (len(imgs) + ncols - 1) // ncols
    aspect = max(img.shape[0] / img.shape[1] for img in imgs)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, (width * aspect + 0.4) * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, img, title in zip(axes.flat, imgs, titles):
        ax.imshow(img)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def video_info(path):
    cap = cv2.VideoCapture(path)
    info = dict(fps=cap.get(cv2.CAP_PROP_FPS), frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    return info


def read_frames(path, max_frames=None, start=0):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    count = 0
    while max_frames is None or count < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        yield cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        count += 1
    cap.release()


def write_video(out_path, frames, fps):
    # frames: orice secvență (sau generator) de cadre RGB
    # WebM (VP9): se redă în browser, în Colab și în VS Code (VS Code nu poate reda fișiere MP4 / H.264)
    with imageio.get_writer(out_path, fps=fps, codec='libvpx-vp9', macro_block_size=16,
                            ffmpeg_params=['-deadline', 'realtime', '-cpu-used', '8', '-b:v', '0', '-crf', '30', '-row-mt', '1']) as writer:
        for frame in frames:
            writer.append_data(frame)
    return out_path


def put_title(img, text, scale=0.7):
    # cv2.putText suportă doar caractere ASCII, deci textele desenate pe imagini nu au diacritice
    cv2.putText(img, text, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img, text, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)
    return img


def synchronize(dev=device):
    if torch.device(dev).type == 'cuda':
        torch.cuda.synchronize()

# Partea I: detecție single-stage vs. two-stage

## 1. Detectoarele
- Faster R-CNN ResNet-50 FPN v2 (two-stage): RPN propune ~1.000 de regiuni, apoi fiecare regiune este clasificată separat. Precis, dar costul crește cu numărul de propuneri.
- YOLO26 n / s / m (single-stage): fiecare poziție din hărțile de trăsături prezice direct clasa și caseta, într-o singură trecere, fără NMS.

Toate modelele sunt pre-antrenate pe COCO (80 de clase) și întorc formatul din laboratorul 6: `boxes`, `labels`, `scores`. `ultralytics` așteaptă imagini BGR, deci inversăm canalele.

Resurse: Faster R-CNN (https://arxiv.org/abs/1506.01497), YOLO (https://arxiv.org/abs/1506.02640), YOLO26 (https://docs.ultralytics.com/models/yolo26/).

In [ ]:
class FasterRCNNDetector:
    def __init__(self, dev=device):
        weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        self.model = fasterrcnn_resnet50_fpn_v2(weights=weights).to(dev).eval()
        self.preprocess = weights.transforms()
        self.dev = dev

    @torch.no_grad()
    def __call__(self, images, threshold=0.5):
        self.model.roi_heads.score_thresh = threshold
        batch = [self.preprocess(torch.from_numpy(img).to(self.dev).permute(2, 0, 1)) for img in images]
        return [dict(boxes=out['boxes'].cpu(), labels=[COCO_CLASSES[i] for i in out['labels'].tolist()], scores=out['scores'].cpu())
                for out in self.model(batch)]


class YOLODetector:
    def __init__(self, weights_name, dev=device):
        self.model = YOLO(os.path.join(DATA_DIR, weights_name))  # descărcat automat la prima utilizare
        self.dev = '0' if torch.device(dev).type == 'cuda' else 'cpu'

    @torch.no_grad()
    def __call__(self, images, threshold=0.5):
        results = self.model.predict([img[..., ::-1] for img in images], conf=threshold, device=self.dev, verbose=False)
        return [dict(boxes=r.boxes.xyxy.cpu(), labels=[r.names[c] for c in r.boxes.cls.int().tolist()], scores=r.boxes.conf.cpu())
                for r in results]


MODELS = {
    'Faster R-CNN': lambda dev: FasterRCNNDetector(dev),
    'YOLO26-n': lambda dev: YOLODetector('yolo26n.pt', dev),
    'YOLO26-s': lambda dev: YOLODetector('yolo26s.pt', dev),
    'YOLO26-m': lambda dev: YOLODetector('yolo26m.pt', dev),
}
detectors = {name: make(device) for name, make in MODELS.items()}


def count_params(model):
    return sum(p.numel() for p in model.parameters())


for name, det in detectors.items():
    model = det.model if name == 'Faster R-CNN' else det.model.model
    print('{:<13} {:>5.1f} milioane de parametri'.format(name, count_params(model) / 1e6))

## 2. Acuratețea: mAP pe COCO val2017
Rulăm fiecare detector o singură dată pe toate imaginile, cu pragul de încredere 0,001 (mAP are nevoie și de detecțiile cu scor mic), și păstrăm detecțiile pentru tot restul Părții I.

O detecție este TP dacă are clasa unei adnotări, IoU >= prag cu ea, iar adnotarea nu a fost deja găsită de o detecție cu scor mai mare (`match_detections`, folosită în aplicația 1). AP este aria de sub curba precizie-recall; mAP COCO este media AP pe clase și pe pragurile IoU 0,5, 0,55, ..., 0,95. Folosim implementarea oficială, `pycocotools`.

In [ ]:
predictions = {}
for name, detector in detectors.items():
    predictions[name] = {image_id: detector([load_image(coco_path(image_id))], threshold=0.001)[0]
                         for image_id in tqdm(IMAGE_IDS, desc=name)}


def gt_annotations(image_id, label=None):
    # adnotările unei imagini (doar ale clasei label, dacă este dată), fără regiunile de tip "crowd"
    cat_ids = [COCO_CLASSES.index(label)] if label else []
    anns = coco.loadAnns(coco.getAnnIds(imgIds=image_id, catIds=cat_ids, iscrowd=False))
    boxes = torch.tensor([a['bbox'] for a in anns], dtype=torch.float32).view(-1, 4)
    boxes = torch.cat([boxes[:, :2], boxes[:, :2] + boxes[:, 2:]], dim=1)  # COCO: (x, y, w, h) -> (x1, y1, x2, y2)
    return dict(boxes=boxes, labels=[COCO_CLASSES[a['category_id']] for a in anns], areas=[a['area'] for a in anns])


def match_detections(det, gt_boxes, iou_threshold=0.5):
    # det și gt_boxes: o singură clasă, o singură imagine
    # întoarce is_tp (pentru fiecare detecție) și found (pentru fiecare adnotare)
    is_tp = torch.zeros(len(det['scores']), dtype=torch.bool)
    found = torch.zeros(len(gt_boxes), dtype=torch.bool)
    if len(gt_boxes) == 0:
        return is_tp, found
    iou = box_iou(det['boxes'], gt_boxes)
    for k in det['scores'].argsort(descending=True).tolist():  # detecțiile sigure aleg primele
        candidates = iou[k].masked_fill(found, 0)               # o adnotare poate fi găsită o singură dată
        j = candidates.argmax()
        if candidates[j] >= iou_threshold:
            found[j] = True
            is_tp[k] = True
    return is_tp, found

In [ ]:
def coco_evaluate(preds):
    results = []
    for image_id, det in preds.items():
        boxes = det['boxes'].clone()
        boxes[:, 2:] -= boxes[:, :2]  # (x1, y1, x2, y2) -> (x, y, w, h)
        results += [dict(image_id=image_id, category_id=COCO_CLASSES.index(label), bbox=box, score=score)
                    for box, label, score in zip(boxes.tolist(), det['labels'], det['scores'].tolist())]
    with contextlib.redirect_stdout(io.StringIO()):
        evaluator = COCOeval(coco, coco.loadRes(results), 'bbox')
        evaluator.params.imgIds = list(preds)
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return evaluator


evaluators = {name: coco_evaluate(predictions[name]) for name in detectors}

In [ ]:
METRICS = ['AP', 'AP50', 'AP75', 'AP small', 'AP medium', 'AP large']
mAP = {name: dict(zip(METRICS, 100 * evaluator.stats[:6])) for name, evaluator in evaluators.items()}

print('{:<13}'.format('') + ''.join('{:>11}'.format(m) for m in METRICS))
for name in detectors:
    print('{:<13}'.format(name) + ''.join('{:>11.1f}'.format(mAP[name][m]) for m in METRICS))

- Faster R-CNN depășește YOLO26-n, mai ales la obiectele mici, dar YOLO26-m are mAP-ul cel mai mare: detectoarele single-stage moderne au recuperat diferența de acuratețe.
- Pe 500 de imagini valorile diferă de cele oficiale (5.000 de imagini), dar ordinea modelelor este aceeași.

## 3. Timpul de inferență
Latența pe imagine (batch = 1): mediana pe mai multe imagini, cu warm-up și `synchronize()`, ca în laboratorul 6. Pe CPU construim instanțe noi ale modelelor.

In [ ]:
def measure_latency(detector, images, dev, warmup=2):
    for img in images[:warmup]:
        detector([img])
    times = []
    for img in images:
        synchronize(dev)
        start = time.perf_counter()
        detector([img])
        synchronize(dev)
        times.append(time.perf_counter() - start)
    return 1000 * np.median(times)  # milisecunde


latency = collections.defaultdict(dict)
sample = [load_image(coco_path(image_id)) for image_id in IMAGE_IDS[:50]]
if device.type == 'cuda':
    for name, detector in detectors.items():
        latency[name]['GPU'] = measure_latency(detector, sample, device)

print('CPU: {} fire de execuție'.format(torch.get_num_threads()))
for name, make in MODELS.items():
    detector = make(torch.device('cpu'))
    latency[name]['CPU'] = measure_latency(detector, sample[:CPU_IMAGES], 'cpu', warmup=1)
    del detector

print('{:<13}{:>14}{:>14}{:>8}'.format('', 'GPU (ms)', 'CPU (ms)', 'mAP'))
for name in detectors:
    print('{:<13}{:>14.1f}{:>14.1f}{:>8.1f}'.format(name, latency[name].get('GPU', float('nan')), latency[name]['CPU'], mAP[name]['AP']))

### 3.1 Compromisul viteză - acuratețe
Un model este mai bun dacă este mai sus (mAP mai mare) și mai la stânga (latență mai mică).

In [ ]:
hardware = [h for h in ['GPU', 'CPU'] if h in latency['Faster R-CNN']]
fig, axes = plt.subplots(1, len(hardware), figsize=(6.5 * len(hardware), 4.5), squeeze=False)
for ax, hw in zip(axes[0], hardware):
    for name in detectors:
        two_stage = name == 'Faster R-CNN'
        ax.scatter(latency[name][hw], mAP[name]['AP'], s=80, marker='s' if two_stage else 'o', color='C3' if two_stage else 'C0')
        ax.annotate(name, (latency[name][hw], mAP[name]['AP']), textcoords='offset points', xytext=(6, 6))
    ax.set_xscale('log')
    ax.set_xlabel('latență pe imagine (ms), {}'.format(hw))
    ax.set_ylabel('mAP COCO (%)')
    ax.set_title('{}: pătrat = two-stage, cerc = single-stage'.format(hw))
    ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

- Pe CPU, doar modelele single-stage mici pot procesa un flux video.
- Pe GPU, la batch = 1, modelele YOLO au latențe apropiate: timpul este dominat de costuri fixe (lansarea operațiilor pe GPU, pre- și post-procesarea).

## 4. Aplicația 1: acuratețea este critică
Analiza offline a unei arhive de imagini (inventar, căutare într-o arhivă de supraveghere): timpul nu contează, dar fiecare obiect ratat costă.

### 4.1 Recall pe dimensiuni de obiecte
Recall (IoU >= 0,5, prag de încredere 0,5) pentru obiectele mici, medii și mari, cu limitele COCO.

In [ ]:
def filter_detections(det, threshold):
    keep = det['scores'] >= threshold
    return dict(boxes=det['boxes'][keep], labels=[l for l, k in zip(det['labels'], keep) if k], scores=det['scores'][keep])


SIZE_GROUPS = {'mici': (0, 32 ** 2), 'medii': (32 ** 2, 96 ** 2), 'mari': (96 ** 2, float('inf'))}


def evaluate_image(det, image_id, iou_threshold=0.5):
    # întoarce is_tp pentru fiecare detecție, found pentru fiecare adnotare și adnotările imaginii
    gt = gt_annotations(image_id)
    is_tp = torch.zeros(len(det['labels']), dtype=torch.bool)
    found = torch.zeros(len(gt['labels']), dtype=torch.bool)
    for label in set(det['labels']) | set(gt['labels']):
        d_idx = torch.tensor([i for i, l in enumerate(det['labels']) if l == label], dtype=torch.long)
        g_idx = torch.tensor([j for j, l in enumerate(gt['labels']) if l == label], dtype=torch.long)
        is_tp[d_idx], found[g_idx] = match_detections(
            dict(boxes=det['boxes'][d_idx], scores=det['scores'][d_idx]), gt['boxes'][g_idx], iou_threshold)
    return is_tp, found, gt


def recall_by_size(preds, threshold=0.5):
    found_by_size = collections.defaultdict(list)
    for image_id, det in preds.items():
        _, found, gt = evaluate_image(filter_detections(det, threshold), image_id)
        for f, area in zip(found.tolist(), gt['areas']):
            for group, (low, high) in SIZE_GROUPS.items():
                if low <= area < high:
                    found_by_size[group].append(f)
    return {group: 100 * np.mean(found_by_size[group]) for group in SIZE_GROUPS}


recalls = {name: recall_by_size(predictions[name]) for name in detectors}
x = np.arange(len(SIZE_GROUPS))
plt.figure(figsize=(8, 4))
for k, name in enumerate(detectors):
    plt.bar(x + (k - 1.5) * 0.2, [recalls[name][g] for g in SIZE_GROUPS], width=0.2, label=name)
plt.xticks(x, ['obiecte {}'.format(g) for g in SIZE_GROUPS])
plt.ylabel('recall (%), prag de încredere 0,5')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.show()

### 4.2 Exemple
Imaginile în care Faster R-CNN găsește cele mai multe obiecte în plus față de YOLO26-n. Verde = TP, roșu = FP, galben = obiect ratat (FN).

In [ ]:
EVAL_COLORS = dict(tp=(30, 200, 30), fp=(230, 30, 30), fn=(255, 220, 0))


def draw_evaluation(image, det, image_id):
    is_tp, found, gt = evaluate_image(det, image_id)
    img = draw_detections(image, det, colors=[EVAL_COLORS['tp'] if t else EVAL_COLORS['fp'] for t in is_tp.tolist()])
    for (x1, y1, x2, y2), f in zip(gt['boxes'].round().int().tolist(), found.tolist()):
        if not f:
            cv2.rectangle(img, (x1, y1), (x2, y2), EVAL_COLORS['fn'], 2)
    title = 'TP {}, FP {}, ratate {} din {}'.format(is_tp.sum().item(), (~is_tp).sum().item(), (~found).sum().item(), len(found))
    return img, title


found_count = {name: {image_id: evaluate_image(filter_detections(predictions[name][image_id], 0.5), image_id)[1].sum().item()
                      for image_id in IMAGE_IDS} for name in ['Faster R-CNN', 'YOLO26-n']}
best = sorted(IMAGE_IDS, key=lambda i: found_count['Faster R-CNN'][i] - found_count['YOLO26-n'][i], reverse=True)[:3]

imgs, titles = [], []
for image_id in best:
    image = load_image(coco_path(image_id))
    for name in ['Faster R-CNN', 'YOLO26-n']:
        img, title = draw_evaluation(image, filter_detections(predictions[name][image_id], 0.5), image_id)
        imgs.append(img)
        titles.append('{}: {}'.format(name, title))
show_images(imgs, titles, ncols=2, width=7)

In [ ]:
ARCHIVE = 100_000
print('Timp pentru o arhivă de {} de imagini, pe GPU:'.format(ARCHIVE))
for name in detectors:
    hours = latency[name].get('GPU', latency[name]['CPU']) / 1000 * ARCHIVE / 3600
    print('  {:<13} {:>5.1f} ore, mAP {:.1f}'.format(name, hours, mAP[name]['AP']))

Concluzie: alegem modelul cu cel mai bun mAP (aici YOLO26-m, apoi Faster R-CNN), indiferent de viteză; arhiva se procesează oricum în câteva ore.

## 5. Aplicația 2: timpul de inferență este critic
Un flux video procesat local, în timp real (robot, mașină, cameră inteligentă), deseori fără GPU.

### 5.1 Simularea unui flux în timp real
- camera produce un cadru la fiecare `1 / fps` secunde;
- detectorul, când este liber, preia cel mai recent cadru și are nevoie de `latență` secunde; cadrele sosite între timp sunt pierdute;
- pe ecran apar cadrul curent și ultimele detecții disponibile.

Detecțiile sunt calculate o singură dată, pe GPU; simularea folosește latențele din secțiunea 3 ca să decidă ce detecții ar fi fost disponibile la fiecare cadru.

In [ ]:
def simulate_stream(n_frames, fps, latency_ms):
    source = np.full(n_frames, -1)
    busy_until, current, last = 0.0, None, -1
    for i in range(n_frames):
        now = i / fps
        if current is not None and busy_until <= now:  # detecția în curs s-a terminat
            last, current = current, None
        if current is None:                            # detectorul este liber: preia cadrul curent
            current, busy_until = i, now + latency_ms / 1000
        source[i] = last
    return source


def detect_video(path, detector, batch_size=4, threshold=0.5):
    frames = read_frames(path)
    detections = []
    with tqdm(total=video_info(path)['frames'], desc=os.path.basename(path)) as progress:
        while True:
            batch = [frame for _, frame in zip(range(batch_size), frames)]
            if not batch:
                break
            detections += detector(batch, threshold)
            progress.update(len(batch))
    return detections


info_street = video_info(VIDEO_STREET)
print(info_street)
video_dets = {name: detect_video(VIDEO_STREET, detectors[name]) for name in ['Faster R-CNN', 'YOLO26-n']}

Acoperirea: fracțiunea din detecțiile pe care modelul le-ar produce pe cadrul curent pentru care ecranul afișează o casetă din aceeași clasă, cu IoU >= 0,5. 100% = model instantaneu; diferența este efectul latenței.

In [ ]:
def coverage(detections, source, iou_threshold=0.5):
    matched, total = 0, 0
    for i, j in enumerate(source):
        fresh = detections[i]
        total += len(fresh['labels'])
        if j < 0 or len(fresh['labels']) == 0:
            continue
        shown = detections[j]
        iou = box_iou(fresh['boxes'], shown['boxes'])
        same = torch.tensor([[a == b for b in shown['labels']] for a in fresh['labels']], dtype=torch.bool).view(iou.shape)
        matched += ((iou * same) >= iou_threshold).any(dim=1).sum().item()
    return 100 * matched / max(total, 1)


n_frames, fps = len(video_dets['Faster R-CNN']), info_street['fps']
streams = {}
print('Camera: {:.0f} cadre pe secundă (un cadru la {:.0f} ms)\n'.format(fps, 1000 / fps))
print('{:<24}{:>12}{:>18}{:>22}{:>13}'.format('', 'latență (ms)', 'cadre procesate', 'vechimea medie (ms)', 'acoperire'))
for hw in hardware:
    for name in video_dets:
        source = simulate_stream(n_frames, fps, latency[name][hw])
        streams[(name, hw)] = source
        valid = source >= 0
        age = 1000 * np.mean((np.arange(n_frames) - source)[valid]) / fps
        print('{:<24}{:>12.0f}{:>17.0f}%{:>22.0f}{:>12.1f}%'.format(
            '{} ({})'.format(name, hw), latency[name][hw], 100 * len(set(source[valid])) / n_frames, age,
            coverage(video_dets[name], source)))

### 5.2 Rezultatul, cadru cu cadru
Fluxul simulat pe CPU: în stânga Faster R-CNN, în dreapta YOLO26-n.

In [ ]:
def render_stream(path, out_path, names, hw):
    fps = video_info(path)['fps']

    def frames():
        for i, frame in enumerate(read_frames(path)):
            panels = []
            for name in names:
                j = streams[(name, hw)][i]
                panel = draw_detections(frame, video_dets[name][j]) if j >= 0 else frame.copy()
                age = 'nicio detectie' if j < 0 else 'detectii vechi de {:.1f} s'.format((i - j) / fps)
                panels.append(put_title(panel, '{} ({}): {}'.format(name, hw, age)))
            yield np.hstack(panels)

    return write_video(out_path, frames(), fps)


hw_demo = 'CPU'
out_stream = render_stream(VIDEO_STREET, os.path.join(OUT_DIR, 'street_realtime_{}.webm'.format(hw_demo)), list(video_dets), hw_demo)
display(Video(out_stream, embed=True, width=1200))

In [ ]:
frame_ids = [200, 330, 570]
frames = {i: f for i, f in enumerate(read_frames(VIDEO_STREET, max(frame_ids) + 1)) if i in frame_ids}
imgs, titles = [], []
for i in frame_ids:
    for name in video_dets:
        j = streams[(name, hw_demo)][i]
        imgs.append(draw_detections(frames[i], video_dets[name][j]) if j >= 0 else frames[i])
        titles.append('cadrul {}, {} ({}): detecții din cadrul {}'.format(i, name, hw_demo, j))
show_images(imgs, titles, ncols=2, width=7)

Concluzie: pe CPU, Faster R-CNN procesează doar câteva cadre, iar detecțiile afișate sunt vechi de secunde; YOLO26-n, deși mai puțin precis, dă cel mai bun rezultat. Regula: întâi bugetul de timp (latența sub intervalul dintre cadre, pe hardware-ul țintă), apoi cel mai bun mAP care îl respectă.

# Partea II: segmentarea imaginilor

## 6. Segmentare semantică de la zero: U-Net pe Oxford-IIIT Pet
Segmentarea semantică atribuie o clasă fiecărui pixel.

Resurse: U-Net (https://arxiv.org/abs/1505.04597), Oxford-IIIT Pet (https://www.robots.ox.ac.uk/~vgg/data/pets/).

### 6.1 Baza de date
7.349 de imagini cu pisici și câini, fiecare cu un trimap: animal, fundal, contur. Folosim 3 clase: fundal, pisică, câine (specia vine din ținta `binary-category`). Conturul primește valoarea 255 și este ignorat la antrenare și la evaluare.

| Split | Imagini | Folosire |
|---|---|---|
| `trainval` | 3.680 | `train` (3.312) + `val` (368) |
| `test` | 3.669 | evaluarea finală |

### 6.2 Încărcarea datelor
Imaginile sunt redimensionate o singură dată la `IMG_SIZE x IMG_SIZE` (biliniar; măștile cu nearest-neighbour) și salvate pe disc.

In [ ]:
SEG_CLASSES = ['fundal', 'pisică', 'câine']
NUM_SEG_CLASSES = len(SEG_CLASSES)
IGNORE = 255


def load_pets(split, size=IMG_SIZE):
    cache = os.path.join(DATA_DIR, 'pets_{}_{}.pt'.format(split, size))
    if os.path.exists(cache):
        return torch.load(cache)
    dataset = datasets.OxfordIIITPet(DATA_ROOT, split=split, target_types=['segmentation', 'binary-category'], download=True)
    images = torch.zeros(len(dataset), 3, size, size, dtype=torch.uint8)
    masks = torch.zeros(len(dataset), size, size, dtype=torch.uint8)
    for i in tqdm(range(len(dataset)), desc=split):
        image, (trimap, species) = dataset[i]
        image = image.convert('RGB').resize((size, size), Image.Resampling.BILINEAR)
        trimap = torch.from_numpy(np.array(trimap.resize((size, size), Image.Resampling.NEAREST)))
        mask = torch.full_like(trimap, IGNORE)  # trimap: 1 = animal, 2 = fundal, 3 = contur (rămâne IGNORE)
        mask[trimap == 2] = 0
        mask[trimap == 1] = 1 + species         # 1 = pisică, 2 = câine
        images[i] = torch.from_numpy(np.array(image)).permute(2, 0, 1)
        masks[i] = mask
    torch.save((images, masks), cache)
    return images, masks


x_trainval, y_trainval = load_pets('trainval')
x_test, y_test = load_pets('test')
train_idx, valid_idx = train_test_split(np.arange(len(x_trainval)), test_size=368, random_state=0)

# Statisticile de normalizare se calculează doar pe train, separat pe fiecare canal
std, mean = torch.std_mean(x_trainval[train_idx].float() / 255, dim=(0, 2, 3))
MEAN, STD = mean.view(1, 3, 1, 1).to(device), std.view(1, 3, 1, 1).to(device)

train_loader = DataLoader(TensorDataset(x_trainval[train_idx], y_trainval[train_idx]), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
valid_loader = DataLoader(TensorDataset(x_trainval[valid_idx], y_trainval[valid_idx]), batch_size=128)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128)

pixels = torch.bincount(y_trainval[train_idx].flatten().long(), minlength=256)
print('train: {}, val: {}, test: {}'.format(len(train_idx), len(valid_idx), len(x_test)))
print('Pixeli în train: ' + ', '.join('{} {:.1f}%'.format(c, 100 * pixels[k] / pixels.sum()) for k, c in enumerate(SEG_CLASSES)) +
      ', contur (ignorat) {:.1f}%'.format(100 * pixels[IGNORE] / pixels.sum()))

In [ ]:
SEG_COLORS = np.array([[0, 0, 0], [255, 140, 0], [40, 120, 255]], dtype=np.uint8)  # fundal, pisică, câine


def colorize(mask):
    # mască H x W de indici de clasă -> imagine RGB; pixelii ignorați devin albi
    mask = np.asarray(mask)
    img = np.full(mask.shape + (3,), 255, dtype=np.uint8)
    valid = mask != IGNORE
    img[valid] = SEG_COLORS[mask[valid]]
    return img


def overlay(image, mask, alpha=0.5):
    return (image * (1 - alpha) + colorize(mask) * alpha).astype(np.uint8)


examples = [0, 1, 2, 3, 4, 5]
imgs = [x_trainval[i].permute(1, 2, 0).numpy() for i in examples]
show_images(imgs + [colorize(y_trainval[i]) for i in examples],
            ['imagine'] * len(examples) + ['mască: negru = fundal, portocaliu = pisică, albastru = câine, alb = contur'] * len(examples),
            ncols=6, width=2.6)

### 6.3 Arhitectura U-Net
- encoder: blocuri de 2 convoluții 3 x 3 + max-pooling; rezoluția se înjumătățește, numărul de canale se dublează;
- decoder: convoluții transpuse 2 x 2 (upsampling) + 2 convoluții 3 x 3;
- conexiuni skip: harta encoder-ului este concatenată cu harta decoder-ului de aceeași rezoluție, pentru detaliile fine pierdute prin pooling;
- convoluția finală 1 x 1 dă scorurile celor 3 clase pentru fiecare pixel.

In [ ]:
def double_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True))


class UNet(nn.Module):
    def __init__(self, num_classes, c=32):
        super(UNet, self).__init__()
        # Intrare: 3x128x128
        self.enc1 = double_conv(3, c)                                     # -> 32x128x128
        self.enc2 = double_conv(c, 2 * c)                                 # -> 64x64x64 (după pooling)
        self.enc3 = double_conv(2 * c, 4 * c)                             # -> 128x32x32
        self.enc4 = double_conv(4 * c, 8 * c)                             # -> 256x16x16
        self.bottleneck = double_conv(8 * c, 16 * c)                      # -> 512x8x8
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.up4 = nn.ConvTranspose2d(16 * c, 8 * c, kernel_size=2, stride=2)  # 512x8x8 -> 256x16x16
        self.dec4 = double_conv(16 * c, 8 * c)                                 # concat cu enc4: 512x16x16 -> 256x16x16
        self.up3 = nn.ConvTranspose2d(8 * c, 4 * c, kernel_size=2, stride=2)   # -> 128x32x32
        self.dec3 = double_conv(8 * c, 4 * c)                                  # concat cu enc3: 256x32x32 -> 128x32x32
        self.up2 = nn.ConvTranspose2d(4 * c, 2 * c, kernel_size=2, stride=2)   # -> 64x64x64
        self.dec2 = double_conv(4 * c, 2 * c)                                  # concat cu enc2: 128x64x64 -> 64x64x64
        self.up1 = nn.ConvTranspose2d(2 * c, c, kernel_size=2, stride=2)       # -> 32x128x128
        self.dec1 = double_conv(2 * c, c)                                      # concat cu enc1: 64x128x128 -> 32x128x128
        self.head = nn.Conv2d(c, num_classes, kernel_size=1)                   # -> num_classes x 128 x 128

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


unet = UNet(NUM_SEG_CLASSES).to(device)
print('Parametri U-Net: {:,}'.format(count_params(unet)))
with torch.no_grad():
    print('Intrare 1x3x{0}x{0} -> ieșire {1}'.format(IMG_SIZE, tuple(unet(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device)).shape)))

### 6.4 Antrenare și evaluare
Cross-entropy pe fiecare pixel (`ignore_index=255`), AdamW cu scădere cosinus, oglindire orizontală aplicată identic imaginii și măștii. Metrici: IoU pe clasă și mIoU, din matricea de confuzie la nivel de pixel; IoU animal unește pisica și câinele.

In [ ]:
def normalize_batch(images):
    return (images.to(device).float() / 255 - MEAN) / STD


@torch.no_grad()
def confusion_matrix(model, loader):
    model.eval()
    cm = torch.zeros(NUM_SEG_CLASSES, NUM_SEG_CLASSES, dtype=torch.long, device=device)
    for images, masks in loader:
        pred = model(normalize_batch(images)).argmax(dim=1)
        masks = masks.to(device).long()
        valid = masks != IGNORE
        cm += torch.bincount(masks[valid] * NUM_SEG_CLASSES + pred[valid], minlength=NUM_SEG_CLASSES ** 2).view(NUM_SEG_CLASSES, NUM_SEG_CLASSES)
    return cm.cpu()


def iou_per_class(cm):
    tp = cm.diag().float()
    return tp / (cm.sum(0) + cm.sum(1) - cm.diag()).float()


def segmentation_metrics(cm):
    animal = torch.tensor([[cm[0, 0], cm[0, 1:].sum()], [cm[1:, 0].sum(), cm[1:, 1:].sum()]])  # pisică + câine -> animal
    iou = iou_per_class(cm)
    return dict(pixel_acc=100 * (cm.diag().sum() / cm.sum()).item(), miou=100 * iou.mean().item(),
                iou=(100 * iou).tolist(), iou_animal=100 * iou_per_class(animal)[1].item())


def train_segmentation(model, epochs, lr=1e-3):
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    history = dict(loss=[], miou=[])
    best_miou, best_state = -1.0, None
    start = time.time()
    for epoch in tqdm(range(epochs), desc='U-Net'):
        model.train()
        running_loss = 0.0
        for images, masks in train_loader:
            images, masks = normalize_batch(images), masks.to(device).long()
            flip = torch.rand(len(images), device=device) < 0.5  # oglindire, aceeași pentru imagine și mască
            images[flip], masks[flip] = images[flip].flip(-1), masks[flip].flip(-1)
            loss = criterion(model(images), masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()
        miou = segmentation_metrics(confusion_matrix(model, valid_loader))['miou']
        history['loss'].append(running_loss / len(train_loader))
        history['miou'].append(miou)
        if miou > best_miou:
            best_miou, best_state = miou, {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0:
            tqdm.write('Epoch [{}/{}], loss: {:.4f}, val mIoU: {:.2f} %'.format(epoch + 1, epochs, history['loss'][-1], miou))
    model.load_state_dict(best_state)
    print('Cel mai bun val mIoU: {:.2f} % (epoca {}), durata: {:.1f} min'.format(
        best_miou, int(np.argmax(history['miou'])) + 1, (time.time() - start) / 60))
    return history


UNET_PATH = os.path.join(CKPT_DIR, 'lab7_unet_pets.pt')
if os.path.exists(UNET_PATH) and not RETRAIN:
    checkpoint = torch.load(UNET_PATH, map_location=device)
    unet.load_state_dict(checkpoint['state_dict'])
    unet_history = checkpoint['history']
    print('Model încărcat din', UNET_PATH)
else:
    unet_history = train_segmentation(unet, EPOCHS)
    torch.save({'state_dict': unet.state_dict(), 'history': unet_history}, UNET_PATH)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(range(1, len(unet_history['loss']) + 1), unet_history['loss'])
axes[0].set_xlabel('epoca')
axes[0].set_ylabel('cross-entropy (train)')
axes[1].plot(range(1, len(unet_history['miou']) + 1), unet_history['miou'])
axes[1].set_xlabel('epoca')
axes[1].set_ylabel('mIoU pe val (%)')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

test_metrics = segmentation_metrics(confusion_matrix(unet, test_loader))
print('Test: acuratețe pixeli {:.1f}%, mIoU {:.1f}%'.format(test_metrics['pixel_acc'], test_metrics['miou']))
print('IoU pe clase: ' + ', '.join('{} {:.1f}%'.format(c, v) for c, v in zip(SEG_CLASSES, test_metrics['iou'])))
print('IoU animal (pisică + câine): {:.1f}%'.format(test_metrics['iou_animal']))

### 6.5 Analiza rezultatelor
Cele mai bune și cele mai slabe 4 imagini de test, după mIoU pe imagine.

In [ ]:
@torch.no_grad()
def predict_masks(model, images, batch_size=128):
    model.eval()
    return torch.cat([model(normalize_batch(images[i:i + batch_size])).argmax(dim=1).cpu()
                      for i in range(0, len(images), batch_size)])


test_pred = predict_masks(unet, x_test)
valid = y_test != IGNORE
correct_animal = ((test_pred > 0) == (y_test > 0)) & valid
per_image_iou = []
for pred, gt, v in zip(test_pred, y_test.long(), valid):
    ious = [((pred == c) & (gt == c) & v).sum() / (((pred == c) | (gt == c)) & v).sum().clamp(min=1) for c in range(NUM_SEG_CLASSES)
            if (gt == c).any() or (pred == c).any()]
    per_image_iou.append(torch.stack(ious).mean().item())
order = np.argsort(per_image_iou)

for title, ids in [('cele mai bune', order[::-1][:4]), ('cele mai slabe', order[:4])]:
    imgs, titles = [], []
    for i in ids:
        image = x_test[i].permute(1, 2, 0).numpy()
        imgs += [image, colorize(y_test[i]), colorize(test_pred[i])]
        titles += ['test {} ({})'.format(i, title), 'adnotare', 'predicție: mIoU {:.0f}%'.format(100 * per_image_iou[i])]
    show_images(imgs, titles, ncols=6, width=2.6)

- Animalul este segmentat bine; erorile principale sunt de recunoaștere a speciei (un encoder pre-antrenat ar ajuta, vezi #TODO).
- Unele dintre cele mai slabe rezultate sunt erori de adnotare (măști inversate sau lipsă).

### 6.6 Cât costă o adnotare?
Adnotați 3 imagini de test și comparați rezultatul cu adnotarea oficială și cu U-Net:
1. `Start`: apare prima imagine și pornește cronometrul;
2. pictați animalul: portocaliu = pisică, albastru = câine; ce rămâne nepictat este fundal;
3. `Trimite`: adnotarea este salvată și apare imaginea următoare.

Masca este redusă la 128 x 128 pixeli, iar banda de contur nu se punctează.

In [ ]:
ANNOTATE_IDS = [25, 108, 75]  # pisică (ușor), câine cu coadă (mediu), pisică într-un copac (greu)
ANNOTATE_SIZE = 384
BRUSH_COLORS = ['#{:02x}{:02x}{:02x}'.format(*SEG_COLORS[c]) for c in (1, 2)]  # pisică, câine

# imaginile originale, la rezoluție mai mare decât cele de 128 x 128, ca să se poată picta comod
pets_test = datasets.OxfordIIITPet(DATA_ROOT, split='test')
annotation_images = [np.array(pets_test[i][0].convert('RGB').resize((ANNOTATE_SIZE, ANNOTATE_SIZE), Image.Resampling.BILINEAR))
                     for i in ANNOTATE_IDS]
annotations = {}  # indicele imaginii de test -> (masca 128 x 128, timpul în secunde)


def strokes_to_mask(layer):
    # layer: stratul pictat, H x W x 4 (RGBA); alpha = 0 unde nu s-a pictat -> fundal
    rgb = layer[..., :3].astype(np.float32)
    dist = np.stack([np.abs(rgb - SEG_COLORS[c]).sum(axis=-1) for c in (1, 2)])  # 2 x H x W
    mask = np.where(layer[..., 3] > 0, 1 + dist.argmin(axis=0), 0).astype(np.uint8)
    return cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)


def image_miou(pred, gt):
    # mIoU pe o singură imagine, ca în secțiunea 6.5: clasele prezente, fără pixelii de contur
    pred, gt = torch.as_tensor(pred).long(), torch.as_tensor(gt).long()
    valid = gt != IGNORE
    ious = [((pred == c) & (gt == c) & valid).sum() / (((pred == c) | (gt == c)) & valid).sum().clamp(min=1)
            for c in range(NUM_SEG_CLASSES) if (gt == c).any() or (pred == c).any()]
    return 100 * torch.stack(ious).float().mean().item()


def start_annotation(state):
    return annotation_images[0], dict(k=0, start=time.time()), 'Imaginea 1 din {}'.format(len(ANNOTATE_IDS))


def submit_annotation(editor, state):
    k = state['k']
    if k < 0 or k >= len(ANNOTATE_IDS):
        return gr.update(), state, 'Apăsați Start.'
    seconds = time.time() - state['start']
    layer = editor['layers'][0] if editor['layers'] else np.zeros((ANNOTATE_SIZE, ANNOTATE_SIZE, 4), np.uint8)
    i = ANNOTATE_IDS[k]
    annotations[i] = (strokes_to_mask(layer), seconds)
    message = 'Imaginea {}: {:.0f} s, mIoU față de adnotarea oficială {:.1f}% (U-Net: {:.1f}%).'.format(
        k + 1, seconds, image_miou(annotations[i][0], y_test[i]), 100 * per_image_iou[i])
    k += 1
    if k == len(ANNOTATE_IDS):
        return gr.update(), dict(k=k, start=None), message + ' Gata! Rulați celula următoare.'
    return annotation_images[k], dict(k=k, start=time.time()), message + ' Urmează imaginea {} din {}.'.format(k + 1, len(ANNOTATE_IDS))


with gr.Blocks() as annotation_app:
    state = gr.State(dict(k=-1, start=None))  # imaginea curentă și momentul în care a apărut
    editor = gr.ImageEditor(type='numpy', sources=[], transforms=[], layers=False, format='png',
                            canvas_size=(ANNOTATE_SIZE, ANNOTATE_SIZE),
                            brush=gr.Brush(colors=[(c, 0.6) for c in BRUSH_COLORS], default_color=(BRUSH_COLORS[0], 0.6),
                                           color_mode='fixed', default_size=12),
                            label='portocaliu = pisică, albastru = câine, nepictat = fundal')
    with gr.Row():
        start_button = gr.Button('Start')
        submit_button = gr.Button('Trimite', variant='primary')
    status = gr.Textbox(label='Rezultat', interactive=False)
    start_button.click(start_annotation, state, [editor, state, status])
    # show_progress='hidden': altfel editorul rămâne blocat în starea 'processing' după schimbarea imaginii
    submit_button.click(submit_annotation, [editor, state], [editor, state, status], show_progress='hidden')

annotation_app.launch()

După adnotare, rulați celula următoare: compară adnotările cu adnotarea oficială și cu U-Net și extrapolează timpul la toată baza de date.

In [ ]:
if not annotations:
    print('Nu există adnotări: folosiți aplicația de mai sus, apoi rulați din nou această celulă.')
else:
    imgs, titles = [], []
    for i, (mask, seconds) in annotations.items():
        imgs += [x_test[i].permute(1, 2, 0).numpy(), colorize(y_test[i]), colorize(mask), colorize(test_pred[i])]
        titles += ['test {}'.format(i), 'adnotarea oficială', 'voi: mIoU {:.0f}%, {:.0f} s'.format(image_miou(mask, y_test[i]), seconds),
                   'U-Net: mIoU {:.0f}%'.format(100 * per_image_iou[i])]
    show_images(imgs, titles, ncols=4, width=3)

    ids = list(annotations)
    synchronize()
    start = time.perf_counter()
    predict_masks(unet, x_test[ids])
    synchronize()
    unet_ms = 1000 * (time.perf_counter() - start) / len(ids)

    seconds = np.mean([s for _, s in annotations.values()])
    print('mIoU mediu: voi {:.1f}%, U-Net {:.1f}%'.format(np.mean([image_miou(m, y_test[i]) for i, (m, _) in annotations.items()]),
                                                       100 * np.mean([per_image_iou[i] for i in ids])))
    print('Timp pe imagine: voi {:.0f} s, U-Net {:.1f} ms'.format(seconds, unet_ms))
    total_hours = seconds * 7349 / 3600
    print('Toate cele 7.349 de imagini Oxford-IIIT Pet, în ritmul vostru: {:.0f} ore, adică {:.0f} zile de lucru (8 ore pe zi)'.format(
        total_hours, total_hours / 8))

- Comparați timpul vostru cu milisecundele U-Net; nici adnotarea manuală nu are mIoU 100% (marginile sunt ambigue).
- Referințe: COCO, ~79 s pentru un obiect conturat cu poligoane (Lin et al., 2014); Cityscapes, peste 1,5 ore pentru o imagine adnotată fin (Cordts et al., 2016).
- Costul scade cu adnotarea asistată: un model (de exemplu SAM) propune masca, omul doar o corectează (vezi #TODO).

## 7. Aplicație: fundal virtual într-un apel video
Segmentare semantică cu 2 clase (persoană / fundal), pe fiecare cadru al camerei.

### 7.1 Modelul
DeepLabV3 cu backbone MobileNetV3-Large, pre-antrenat (torchvision, 21 de clase VOC). Păstrăm probabilitatea clasei `person`; cadrul este redus la lățimea `SEG_WIDTH` pentru viteză.

In [ ]:
seg_weights = DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
seg_model = deeplabv3_mobilenet_v3_large(weights=seg_weights).to(device).eval()
VOC_CLASSES = seg_weights.meta['categories']
PERSON = VOC_CLASSES.index('person')
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
SEG_WIDTH = 512

print('Parametri: {:.1f} milioane'.format(count_params(seg_model) / 1e6))
print('Clase:', VOC_CLASSES)


@torch.no_grad()
def segment_voc(frame, width=SEG_WIDTH):
    # întoarce probabilitățile celor 21 de clase VOC, la rezoluția cadrului: 21 x H x W
    h, w = frame.shape[:2]
    x = torch.from_numpy(frame).to(device).permute(2, 0, 1)[None].float() / 255
    x = F.interpolate(x, size=(round(h * width / w), width), mode='bilinear', align_corners=False)
    probs = seg_model((x - IMAGENET_MEAN) / IMAGENET_STD)['out'].softmax(dim=1)
    return F.interpolate(probs, size=(h, w), mode='bilinear', align_corners=False)[0]


def person_probability(frame):
    return segment_voc(frame)[PERSON].cpu().numpy()  # H x W, valori în [0, 1]


call_frames = {i: f for i, f in enumerate(read_frames(VIDEO_CALL, 760)) if i in (150, 450, 750)}
show_images([f for f in call_frames.values()] + [person_probability(f) for f in call_frames.values()],
            ['cadrul {}'.format(i) for i in call_frames] + ['P(person)'] * 3, ncols=3, width=5)

### 7.2 Compunerea imaginii
`rezultat = alpha * cadru + (1 - alpha) * fundal_nou`, cu `alpha` = probabilitatea `person`, ușor încețoșată (feathering), ca marginile să fie netede.

Fundaluri: `blur`, `green` și temele din `BACKGROUNDS` (imaginile `laborator/images/lab7_bg_*`, descărcate din repository dacă lipsesc local).

In [ ]:
BACKGROUND_URL = 'https://raw.githubusercontent.com/MihaiDogariu/CV3/main/laborator/images/{}'
BACKGROUND_FILES = {'my little pony': 'lab7_bg_pony.jpg', 'Mordor': 'lab7_bg_mordor.jpg',
                    'Hogwarts': 'lab7_bg_hogwarts.png', 'office': 'lab7_bg_office.jpg'}


def load_background(name):
    path = os.path.join('images', name)  # local, în repository
    if not os.path.exists(path):
        path = os.path.join(DATA_DIR, name)
        if not os.path.exists(path):
            torch.hub.download_url_to_file(BACKGROUND_URL.format(name), path, progress=False)
    return load_image(path)


BACKGROUNDS = {theme: load_background(name) for theme, name in BACKGROUND_FILES.items()}


def fit_background(image, h, w):
    # decupare centrală la proporțiile h:w, apoi redimensionare
    ih, iw = image.shape[:2]
    scale = max(h / ih, w / iw)
    resized = cv2.resize(image, (round(iw * scale), round(ih * scale)), interpolation=cv2.INTER_AREA)
    y, x = (resized.shape[0] - h) // 2, (resized.shape[1] - w) // 2
    return resized[y:y + h, x:x + w]


def make_background(frame, mode):
    if mode == 'blur':
        # încețoșarea la rezoluție completă este lentă (filtrul are 90 x 90 de pixeli); o facem la 1/4 din rezoluție
        h, w = frame.shape[:2]
        small = cv2.resize(frame, (w // 4, h // 4), interpolation=cv2.INTER_AREA)
        return cv2.resize(cv2.GaussianBlur(small, (0, 0), sigmaX=4), (w, h), interpolation=cv2.INTER_LINEAR)
    if mode in BACKGROUNDS:
        return fit_background(BACKGROUNDS[mode], *frame.shape[:2])
    if mode == 'green':
        return np.full_like(frame, (0, 177, 64))
    return frame


def composite(frame, alpha, mode):
    if mode == 'mask':
        return np.repeat((alpha[..., None] * 255).astype(np.uint8), 3, axis=2)
    alpha = alpha.astype(np.float32)
    return cv2.blendLinear(frame, make_background(frame, mode), alpha, 1 - alpha)  # alpha * cadru + (1 - alpha) * fundal


frame = call_frames[450]
prob = person_probability(frame)
variants = {'binar (alpha > 0,5)': (prob > 0.5).astype(np.float32), 'probabilități': prob,
            'probabilități + feathering': cv2.GaussianBlur(prob, (0, 0), sigmaX=3)}
y0, x0 = 120, 120  # zona mărită: conturul capului și umerilor persoanei din stânga
show_images([composite(frame, a, 'green')[y0:y0 + 160, x0:x0 + 220] for a in variants.values()], list(variants), ncols=3, width=5)
modes = ['blur', 'green'] + list(BACKGROUNDS)
show_images([composite(frame, variants['probabilități + feathering'], m) for m in modes], modes, ncols=3, width=5)

### 7.3 Stabilitatea în timp
Netezire exponențială a măștii, ca să nu "fiarbă" pe margini: `alpha_t = s * alpha_(t-1) + (1 - s) * alpha_nou`. `BackgroundReplacer` adună segmentarea, feathering-ul, netezirea și compunerea.

In [ ]:
class BackgroundReplacer:
    def __init__(self, mode='blur', smoothing=0.5, feather=3):
        self.mode = mode
        self.smoothing = smoothing
        self.feather = feather
        self.alpha = None

    def update_alpha(self, frame, prob=None):
        alpha = person_probability(frame) if prob is None else prob
        if self.feather > 0:
            alpha = cv2.GaussianBlur(alpha, (0, 0), sigmaX=self.feather)
        if self.alpha is not None and self.alpha.shape == alpha.shape:
            alpha = self.smoothing * self.alpha + (1 - self.smoothing) * alpha
        self.alpha = alpha
        return alpha

    def __call__(self, frame):
        return composite(frame, self.update_alpha(frame), self.mode)


CALL_START, CALL_FRAMES = 48, 480  # o persoană (secundele 4-19), apoi un hol gol și două persoane (28-44)
call_info = video_info(VIDEO_CALL)
call_probs = [person_probability(f) for f in tqdm(read_frames(VIDEO_CALL, CALL_FRAMES, CALL_START), total=CALL_FRAMES)]

for s in [0.0, 0.5, 0.8]:
    replacer = BackgroundReplacer(smoothing=s)
    alphas = [replacer.update_alpha(None, prob) for prob in call_probs]
    flicker = np.mean([np.abs(a - b).mean() for a, b in zip(alphas[1:], alphas[:-1])])
    print('netezire s = {}: pâlpâire medie {:.4f}'.format(s, flicker))

### 7.4 Aplicația pe un videoclip
Panouri: original, mască, `blur`, `office`. La final, viteza întregului lanț (segmentare + compunere).

In [ ]:
def half(img):
    return cv2.resize(img, (img.shape[1] // 2, img.shape[0] // 2), interpolation=cv2.INTER_AREA)


def render_background():
    replacer = BackgroundReplacer(smoothing=0.5)
    for frame, prob in zip(read_frames(VIDEO_CALL, CALL_FRAMES, CALL_START), call_probs):
        alpha = replacer.update_alpha(frame, prob)
        panels = [put_title(half(composite(frame, alpha, m)), m, 0.5) for m in ['original', 'mask', 'blur', 'office']]
        yield np.vstack([np.hstack(panels[:2]), np.hstack(panels[2:])])


out_bg = write_video(os.path.join(OUT_DIR, 'call_background.webm'), render_background(), call_info['fps'])
display(Video(out_bg, embed=True, width=768))

replacer = BackgroundReplacer('blur')
test_frames = list(read_frames(VIDEO_CALL, 50, 100))
replacer(test_frames[0])  # warm-up
synchronize()
start = time.perf_counter()
for f in test_frames:
    replacer(f)
synchronize()
print('{:.1f} cadre pe secundă pe {}, la {} x {} pixeli'.format(len(test_frames) / (time.perf_counter() - start), device, *test_frames[0].shape[1::-1]))

### 7.5 Aplicația live
Camera este citită de browser (Gradio), deci aplicația merge și în Colab. Permiteți accesul la cameră, porniți înregistrarea și alegeți fundalul din listă.

In [ ]:
background_app = BackgroundReplacer('office')


def live_background(frame, theme):
    if frame is None:
        return None
    background_app.mode = theme
    return background_app(frame)


with gr.Blocks() as live_app:
    theme = gr.Dropdown(list(BACKGROUNDS), value='office', label='Fundal')
    with gr.Row():
        webcam = gr.Image(sources=['webcam'], streaming=True, webcam_options=gr.WebcamOptions(mirror=True), label='Camera')
        output = gr.Image(label='Fundal virtual')
    webcam.stream(live_background, [webcam, theme], output, stream_every=0.05, concurrency_limit=1)

live_app.launch()

## 8. Instanțe și tracking
Resurse: Mask R-CNN (https://arxiv.org/abs/1703.06870), segmentare panoptică (https://arxiv.org/abs/1801.00868), ByteTrack (https://arxiv.org/abs/2110.06864).

### 8.1 Limita segmentării semantice
Segmentarea semantică nu separă persoanele: toate formează o singură regiune `person`.
- semantică: o clasă pe pixel;
- de instanțe: o mască pentru fiecare obiect numărabil;
- panoptică: fiecare pixel primește o clasă și un id de instanță (things = obiecte numărabile, stuff = regiuni amorfe).

In [ ]:
group_frame = next(read_frames(VIDEO_CALL, 1, 750))
semantic = segment_voc(group_frame).argmax(dim=0).cpu().numpy()
VOC_PALETTE = (np.array(plt.cm.tab20(np.linspace(0, 1, len(VOC_CLASSES)))[:, :3]) * 255).astype(np.uint8)
VOC_PALETTE[0] = 0
show_images([group_frame, VOC_PALETTE[semantic]],
            ['cadrul 750', 'segmentare semantică: clase prezente = {}'.format([VOC_CLASSES[c] for c in np.unique(semantic)])], ncols=2, width=7)

### 8.2 Segmentarea instanțelor: Mask R-CNN
Faster R-CNN plus o ramură care prezice, pentru fiecare detecție, o mască de 28 x 28 pixeli în interiorul casetei.

In [ ]:
mrcnn_weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
mrcnn = maskrcnn_resnet50_fpn_v2(weights=mrcnn_weights).to(device).eval()


@torch.no_grad()
def detect_instances(image, threshold=0.5):
    x = mrcnn_weights.transforms()(torch.from_numpy(image).to(device).permute(2, 0, 1))
    out = mrcnn([x])[0]
    keep = out['scores'] >= threshold
    return dict(boxes=out['boxes'][keep].cpu(), labels=[COCO_CLASSES[i] for i in out['labels'][keep].tolist()],
                scores=out['scores'][keep].cpu(), masks=out['masks'][keep, 0].cpu().numpy())  # măști N x H x W, valori în [0, 1]


def draw_instances(image, inst, alpha=0.5):
    img = image.astype(np.float32)
    for k, mask in enumerate(inst['masks']):
        color = np.array(PALETTE[(2 * k) % len(PALETTE)], dtype=np.float32)
        m = (mask >= 0.5)[..., None]
        img = np.where(m, img * (1 - alpha) + color * alpha, img)
    return draw_detections(img.astype(np.uint8), inst, colors=[PALETTE[(2 * k) % len(PALETTE)] for k in range(len(inst['labels']))])


group_instances = detect_instances(group_frame)
show_images([draw_instances(group_frame, group_instances)], ['Mask R-CNN: {} instanțe'.format(len(group_instances['labels']))], ncols=1, width=8)

### 8.3 Harta panoptică
Regula euristică din Kirillov et al. (2019): instanțele se plasează în ordinea scorului (o instanță acoperită peste 50% de altele este eliminată), iar pixelii rămași primesc clasa semantică, dacă aceasta este stuff; restul rămân void (id 0).

In [ ]:
def panoptic_merge(semantic, instances, stuff_classes=('__background__',), overlap_threshold=0.5, mask_threshold=0.5):
    panoptic = np.zeros(semantic.shape, dtype=np.int32)  # 0 = void
    segments = []
    for k in np.argsort(-instances['scores'].numpy()):
        mask = instances['masks'][k] >= mask_threshold
        free = mask & (panoptic == 0)
        if mask.sum() == 0 or free.sum() / mask.sum() < overlap_threshold:
            continue
        segments.append(dict(id=len(segments) + 1, label=instances['labels'][k], thing=True))
        panoptic[free] = segments[-1]['id']
    for c in np.unique(semantic[panoptic == 0]):
        if VOC_CLASSES[c] in stuff_classes:
            segments.append(dict(id=len(segments) + 1, label=VOC_CLASSES[c], thing=False))
            panoptic[(panoptic == 0) & (semantic == c)] = segments[-1]['id']
    return panoptic, segments


def draw_panoptic(image, panoptic, segments, alpha=0.6):
    colors = np.zeros((panoptic.max() + 1, 3), dtype=np.float32)  # void rămâne negru
    for s in segments:
        colors[s['id']] = PALETTE[(3 * s['id']) % len(PALETTE)]
    img = (image * (1 - alpha) + colors[panoptic] * alpha).astype(np.uint8)
    for s in segments:
        ys, xs = np.nonzero(panoptic == s['id'])
        text = '{}{}'.format(s['label'], ' #{}'.format(s['id']) if s['thing'] else '')
        org = (int(np.median(xs)) - 30, int(np.median(ys)))  # textul se scrie în centrul segmentului
        cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img


group_panoptic, group_segments = panoptic_merge(semantic, group_instances)
for s in group_segments:
    print('segment {}: {:<15} {:<6} {:>6} pixeli'.format(s['id'], s['label'], 'thing' if s['thing'] else 'stuff', (group_panoptic == s['id']).sum()))
print('void: {} pixeli'.format((group_panoptic == 0).sum()))
show_images([draw_panoptic(group_frame, group_panoptic, group_segments)], ['hartă panoptică (DeepLabV3 + Mask R-CNN)'], ncols=1, width=8)

### 8.4 Tracking pe instanțe: câte un apel pentru fiecare persoană
Combinăm segmentarea de instanțe cu tracking-ul din laboratorul 6, folosind modele off-the-shelf: YOLO26-seg (detecție + măști) și tracker-ul ByteTrack din `ultralytics` (`model.track(..., persist=True)`). Fiecare persoană primește un id, păstrat de la un cadru la altul.

Păstrăm cele 3 id-uri urmărite cel mai mult timp. Pentru fiecare, păstrăm în cadru doar persoana respectivă:

`alpha = P(person) * dilatare(masca id-ului)`

Masca instanței alege persoana, iar probabilitatea DeepLabV3 dă conturul fin.

In [ ]:
seg_tracker = YOLO(os.path.join(DATA_DIR, 'yolo26m-seg.pt'))  # descărcat automat la prima utilizare
GROUP_START, GROUP_FRAMES = 640, 210  # cele 3 persoane intră în cadru, se apropie de cameră și ies


def track_persons(frames):
    # pentru fiecare cadru: {id: masca binară H x W}
    tracks = []
    for frame in tqdm(frames, total=GROUP_FRAMES, desc='tracking'):
        r = seg_tracker.track(frame[..., ::-1], persist=True, classes=[0], tracker='bytetrack.yaml', retina_masks=True,
                              device=0 if device.type == 'cuda' else 'cpu', verbose=False)[0]
        ids = [] if r.boxes.id is None else r.boxes.id.int().tolist()
        masks = [] if r.masks is None else r.masks.data.cpu().numpy() > 0.5  # retina_masks: la rezoluția cadrului
        tracks.append(dict(zip(ids, masks)))
    return tracks


def draw_tracks(frame, tracks, alpha=0.5):
    img = frame.copy()
    for i, mask in tracks.items():
        color = PALETTE[(2 * i) % len(PALETTE)]
        img[mask] = (img[mask] * (1 - alpha) + np.array(color) * alpha).astype(np.uint8)
        ys, xs = np.nonzero(mask)
        cv2.putText(img, 'id {}'.format(i), (int(xs.mean()) - 20, int(ys.min()) + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(img, 'id {}'.format(i), (int(xs.mean()) - 20, int(ys.min()) + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    return img


group_tracks = track_persons(read_frames(VIDEO_CALL, GROUP_FRAMES, GROUP_START))
frames_per_id = collections.Counter(i for t in group_tracks for i in t)
print('id: număr de cadre', dict(frames_per_id))
PERSON_IDS = sorted(i for i, _ in frames_per_id.most_common(3))

shown = [50, 110, 160]
group_frames = {k: f for k, f in enumerate(read_frames(VIDEO_CALL, max(shown) + 1, GROUP_START)) if k in shown}
show_images([draw_tracks(group_frames[k], group_tracks[k]) for k in shown], ['cadrul {}'.format(GROUP_START + k) for k in shown], ncols=3, width=5)

Videoclipul are 4 panouri: originalul și câte un apel pentru fiecare persoană urmărită. Când persoana nu este în cadru, panoul ei rămâne gol.

In [ ]:
DILATE = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))


def render_tracking():
    apps = {i: BackgroundReplacer('office') for i in PERSON_IDS}  # câte o netezire în timp pentru fiecare persoană
    for frame, tracks in zip(read_frames(VIDEO_CALL, GROUP_FRAMES, GROUP_START), group_tracks):
        prob = person_probability(frame)
        panels = [put_title(half(frame), 'original', 0.5)]
        for i in PERSON_IDS:
            mask = cv2.dilate(tracks[i].astype(np.uint8), DILATE) if i in tracks else np.zeros(prob.shape, np.uint8)
            alpha = apps[i].update_alpha(frame, prob * mask)
            panels.append(put_title(half(composite(frame, alpha, 'office')), 'id {}'.format(i), 0.5))
        yield np.vstack([np.hstack(panels[:2]), np.hstack(panels[2:])])


out_tracking = write_video(os.path.join(OUT_DIR, 'call_tracking.webm'), render_tracking(), call_info['fps'])
display(Video(out_tracking, embed=True, width=768))

# #TODO:
1. mAP: rulați evaluarea COCO din secțiunea 2 cu detecțiile filtrate la pragul de încredere 0,5 (`filter_detections`), în loc de 0,001. De ce scade mAP? Ce parte a curbei precizie-recall se pierde?
2. Aplicația 2: repetați simularea din secțiunea 5.1 pentru o cameră cu 30 și 60 de cadre pe secundă, cu latențele de pe GPU. De la ce frecvență nu mai este Faster R-CNN utilizabil? Cum ați putea compensa vechimea detecțiilor folosind tracker-ul din laboratorul 6?
3. Dați exemple de 2 aplicații în care se preferă utilizarea modelului în 2 etape și 2 aplicații în care se preferă utilizarea modelului într-o singură etapă.
4. U-Net: eliminați conexiunile skip (decoder-ul primește doar ieșirea `up`, fără concatenare; ajustați numărul de canale) și reantrenați. Cum se schimbă IoU animal și cum arată marginile măștilor? Explicați rolul conexiunilor skip.
5. U-Net: înlocuiți encoder-ul cu un ResNet-18 pre-antrenat pe ImageNet (laboratorul 5) și comparați IoU pentru pisică și câine. Ce problemă rezolvă pre-antrenarea: segmentarea sau recunoașterea speciei?
6. Adnotare asistată: încărcați modelul SAM din `ultralytics` (`SAM('sam2.1_b.pt')`) și obțineți masca animalului din imaginile din secțiunea 6.6 cu un singur click (parametrul `points`). Adăugați în aplicația de adnotare un buton care pornește de la masca propusă de SAM, pe care o corectați apoi cu pensula. Cât timp economisiți și cum se schimbă mIoU?
7. Tracking: rulați secțiunea 8.4 cu tracker-ul BoT-SORT (`tracker='botsort.yaml'`) și cu YOLO26-n-seg. Câte id-uri apar și în ce cazuri se schimbă id-ul unei persoane?